# 📊 Chapter 4: Data Analysis Workflow in Jupyter

A **complete data analysis project** using only Python's **standard library**.

We'll create a synthetic sales dataset, then analyze it to find:

| Analysis | Technique |
|----------|-----------|
| Top products by revenue | `csv` module + manual aggregation |
| Monthly trends | `datetime` + grouping |
| Category breakdowns | `collections.Counter` |
| Statistical summary | `statistics` module |

> 💡 **No pandas required!** This chapter proves you can do real analysis
> with just the standard library.

---
## 4.1 Creating a Synthetic Sales Dataset

We'll generate a CSV file with **500 sales transactions** spanning 12 months.

In [ ]:
import csv
import random
import datetime
import os

# Seed for reproducibility
random.seed(42)

# Define our product catalog
PRODUCTS = {
    'Electronics': [
        ('Laptop', 800, 1500),
        ('Smartphone', 400, 1200),
        ('Headphones', 30, 300),
        ('Tablet', 300, 800),
        ('Smart Watch', 150, 500),
    ],
    'Office': [
        ('Desk Chair', 150, 600),
        ('Monitor', 200, 800),
        ('Keyboard', 20, 150),
        ('Mouse', 10, 80),
        ('Webcam', 30, 200),
    ],
    'Software': [
        ('Antivirus', 20, 80),
        ('Office Suite', 50, 200),
        ('Cloud Storage', 5, 30),
        ('VPN Service', 5, 15),
        ('Design Tool', 30, 100),
    ]
}

REGIONS = ['North', 'South', 'East', 'West']


def generate_sales_data(num_records=500):
    """Generate synthetic sales data."""
    records = []
    start_date = datetime.date(2024, 1, 1)
    end_date = datetime.date(2024, 12, 31)
    date_range = (end_date - start_date).days
    
    for i in range(1, num_records + 1):
        # Random date in 2024
        sale_date = start_date + datetime.timedelta(days=random.randint(0, date_range))
        
        # Random product
        category = random.choice(list(PRODUCTS.keys()))
        product_name, price_min, price_max = random.choice(PRODUCTS[category])
        
        # Random price and quantity
        unit_price = round(random.uniform(price_min, price_max), 2)
        quantity = random.randint(1, 10)
        total = round(unit_price * quantity, 2)
        
        # Random region
        region = random.choice(REGIONS)
        
        records.append({
            'order_id': f'ORD-{i:04d}',
            'date': sale_date.isoformat(),
            'product': product_name,
            'category': category,
            'unit_price': unit_price,
            'quantity': quantity,
            'total': total,
            'region': region
        })
    
    return records


# Generate and save to CSV
sales_data = generate_sales_data(500)

csv_file = 'sales_data_2024.csv'
fieldnames = ['order_id', 'date', 'product', 'category', 'unit_price', 'quantity', 'total', 'region']

with open(csv_file, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sales_data)

print(f'✅ Created {csv_file} with {len(sales_data)} records')
print(f'   File size: {os.path.getsize(csv_file):,} bytes')

---
## 4.2 Reading and Exploring the Data

Let's load the CSV and do some initial exploration.

In [ ]:
import csv

# Read the CSV file
def read_csv(filepath):
    """Read a CSV file and return a list of dicts."""
    with open(filepath, newline='') as f:
        reader = csv.DictReader(f)
        return list(reader)


data = read_csv('sales_data_2024.csv')

# Convert numeric fields
for row in data:
    row['unit_price'] = float(row['unit_price'])
    row['quantity'] = int(row['quantity'])
    row['total'] = float(row['total'])

print(f'Total records: {len(data)}')
print(f'Fields: {list(data[0].keys())}')
print(f'\nFirst 5 records:')
for row in data[:5]:
    print(f"  {row['order_id']}  {row['date']}  {row['product']:15s}  "
          f"${row['total']:>9,.2f}  ({row['region']})")

In [ ]:
# Basic statistics about the dataset
import statistics

totals = [row['total'] for row in data]
quantities = [row['quantity'] for row in data]

print('='*55)
print('          DATASET OVERVIEW')
print('='*55)
print(f'  Records:          {len(data):>10,}')
print(f'  Total Revenue:    ${sum(totals):>12,.2f}')
print(f'  Avg Order Value:  ${statistics.mean(totals):>12,.2f}')
print(f'  Median Order:     ${statistics.median(totals):>12,.2f}')
print(f'  Std Dev:          ${statistics.stdev(totals):>12,.2f}')
print(f'  Min Order:        ${min(totals):>12,.2f}')
print(f'  Max Order:        ${max(totals):>12,.2f}')
print(f'  Avg Quantity:     {statistics.mean(quantities):>12.1f}')
print('='*55)

---
## 4.3 Top Products by Revenue

In [ ]:
from collections import defaultdict

# Aggregate revenue by product
product_revenue = defaultdict(float)
product_count = defaultdict(int)
product_qty = defaultdict(int)

for row in data:
    product = row['product']
    product_revenue[product] += row['total']
    product_count[product] += 1
    product_qty[product] += row['quantity']

# Sort by revenue descending
sorted_products = sorted(product_revenue.items(), key=lambda x: x[1], reverse=True)

total_revenue = sum(product_revenue.values())

print('🏆 TOP PRODUCTS BY REVENUE')
print('=' * 70)
print(f'{"Rank":>4} {"Product":15s} {"Revenue":>12s} {"Orders":>7s} {"Qty":>6s} {"% Total":>8s}')
print('-' * 70)

for rank, (product, revenue) in enumerate(sorted_products, 1):
    orders = product_count[product]
    qty = product_qty[product]
    pct = revenue / total_revenue * 100
    bar = '█' * int(pct / 2)
    print(f'{rank:>4}. {product:15s} ${revenue:>11,.2f} {orders:>7d} {qty:>6d} {pct:>7.1f}% {bar}')

print('-' * 70)
print(f'     {"TOTAL":15s} ${total_revenue:>11,.2f}')

---
## 4.4 Monthly Revenue Trends

In [ ]:
from collections import defaultdict
import datetime

# Group by month
monthly_revenue = defaultdict(float)
monthly_orders = defaultdict(int)

for row in data:
    date = datetime.date.fromisoformat(row['date'])
    month_key = f'{date.year}-{date.month:02d}'
    monthly_revenue[month_key] += row['total']
    monthly_orders[month_key] += 1

# Sort by month
months_sorted = sorted(monthly_revenue.keys())

month_names = {
    '01': 'Jan', '02': 'Feb', '03': 'Mar', '04': 'Apr',
    '05': 'May', '06': 'Jun', '07': 'Jul', '08': 'Aug',
    '09': 'Sep', '10': 'Oct', '11': 'Nov', '12': 'Dec'
}

max_rev = max(monthly_revenue.values())

print('📈 MONTHLY REVENUE TRENDS (2024)')
print('=' * 65)

for month in months_sorted:
    rev = monthly_revenue[month]
    orders = monthly_orders[month]
    mm = month.split('-')[1]
    name = month_names.get(mm, mm)
    bar_len = int(rev / max_rev * 30)
    bar = '█' * bar_len + '░' * (30 - bar_len)
    print(f'  {name} │ {bar} ${rev:>10,.2f}  ({orders} orders)')

print('=' * 65)
print(f'  Total Revenue: ${sum(monthly_revenue.values()):>,.2f}')

---
## 4.5 Category Breakdowns

In [ ]:
from collections import Counter, defaultdict
import statistics

# Aggregate by category
cat_revenue = defaultdict(float)
cat_orders = defaultdict(int)
cat_totals = defaultdict(list)

for row in data:
    cat = row['category']
    cat_revenue[cat] += row['total']
    cat_orders[cat] += 1
    cat_totals[cat].append(row['total'])

total_rev = sum(cat_revenue.values())

print('📦 CATEGORY BREAKDOWN')
print('=' * 75)
print(f'{"Category":15s} {"Revenue":>12s} {"% Share":>8s} {"Orders":>7s} '
      f'{"Avg Order":>10s} {"Median":>10s}')
print('-' * 75)

for cat in sorted(cat_revenue, key=cat_revenue.get, reverse=True):
    rev = cat_revenue[cat]
    pct = rev / total_rev * 100
    orders = cat_orders[cat]
    avg = statistics.mean(cat_totals[cat])
    med = statistics.median(cat_totals[cat])
    
    print(f'{cat:15s} ${rev:>11,.2f} {pct:>7.1f}% {orders:>7d} '
          f'${avg:>9,.2f} ${med:>9,.2f}')

print('-' * 75)

In [ ]:
# Frequency analysis with Counter
from collections import Counter

# Product frequency
product_freq = Counter(row['product'] for row in data)

print('🔢 PRODUCT ORDER FREQUENCY')
print('=' * 50)
for product, count in product_freq.most_common():
    bar = '▓' * (count // 2)
    print(f'  {product:15s} │ {count:>3d} │ {bar}')

print()

# Region frequency
region_freq = Counter(row['region'] for row in data)

print('🗺️  ORDERS BY REGION')
print('=' * 40)
for region, count in region_freq.most_common():
    bar = '▓' * (count // 3)
    print(f'  {region:10s} │ {count:>3d} │ {bar}')

---
## 4.6 Regional Revenue Analysis

In [ ]:
from collections import defaultdict
import statistics

# Revenue by region
region_revenue = defaultdict(float)
region_totals = defaultdict(list)

# Revenue by region × category
region_cat_revenue = defaultdict(lambda: defaultdict(float))

for row in data:
    region = row['region']
    region_revenue[region] += row['total']
    region_totals[region].append(row['total'])
    region_cat_revenue[region][row['category']] += row['total']

total_rev = sum(region_revenue.values())

print('🗺️  REVENUE BY REGION')
print('=' * 60)

for region in sorted(region_revenue, key=region_revenue.get, reverse=True):
    rev = region_revenue[region]
    pct = rev / total_rev * 100
    avg = statistics.mean(region_totals[region])
    n = len(region_totals[region])
    
    print(f'\n  📍 {region} Region')
    print(f'     Revenue:  ${rev:>12,.2f}  ({pct:.1f}% of total)')
    print(f'     Orders:   {n:>12d}')
    print(f'     Avg Order: ${avg:>11,.2f}')
    
    # Category breakdown within region
    print(f'     Categories:')
    for cat in sorted(region_cat_revenue[region],
                      key=region_cat_revenue[region].get, reverse=True):
        cat_rev = region_cat_revenue[region][cat]
        cat_pct = cat_rev / rev * 100
        print(f'       {cat:12s} ${cat_rev:>10,.2f}  ({cat_pct:.0f}%)')

print(f'\n{"=" * 60}')

---
## 4.7 Statistical Deep Dive

In [ ]:
import statistics
import math

totals = [row['total'] for row in data]

# Compute comprehensive statistics
stats = {
    'count': len(totals),
    'sum': sum(totals),
    'mean': statistics.mean(totals),
    'median': statistics.median(totals),
    'mode': statistics.mode([round(t, -2) for t in totals]),  # rounded mode
    'stdev': statistics.stdev(totals),
    'variance': statistics.variance(totals),
    'min': min(totals),
    'max': max(totals),
    'range': max(totals) - min(totals),
}

# Quartiles
sorted_totals = sorted(totals)
n = len(sorted_totals)
q1 = statistics.median(sorted_totals[:n//2])
q3 = statistics.median(sorted_totals[n//2 + (n % 2):])
iqr = q3 - q1

print('📊 COMPREHENSIVE STATISTICS')
print('=' * 45)
print(f'  Count:      {stats["count"]:>12,}')
print(f'  Sum:        ${stats["sum"]:>12,.2f}')
print(f'  Mean:       ${stats["mean"]:>12,.2f}')
print(f'  Median:     ${stats["median"]:>12,.2f}')
print(f'  Mode (≈):   ${stats["mode"]:>12,.2f}')
print(f'  Std Dev:    ${stats["stdev"]:>12,.2f}')
print(f'  Variance:   ${stats["variance"]:>12,.2f}')
print(f'  Min:        ${stats["min"]:>12,.2f}')
print(f'  Max:        ${stats["max"]:>12,.2f}')
print(f'  Range:      ${stats["range"]:>12,.2f}')
print(f'  Q1:         ${q1:>12,.2f}')
print(f'  Q3:         ${q3:>12,.2f}')
print(f'  IQR:        ${iqr:>12,.2f}')
print('=' * 45)

# Distribution (histogram buckets)
print('\n📊 ORDER VALUE DISTRIBUTION')
print('=' * 55)

bucket_size = 500
max_val = int(max(totals)) + bucket_size
buckets = defaultdict(int)

for t in totals:
    bucket = int(t // bucket_size) * bucket_size
    buckets[bucket] += 1

max_count = max(buckets.values())

for bucket_start in sorted(buckets.keys()):
    count = buckets[bucket_start]
    bar = '█' * int(count / max_count * 35)
    label = f'${bucket_start:>5,} - ${bucket_start + bucket_size - 1:>5,}'
    print(f'  {label} │ {bar} {count}')

---
## 📝 Chapter 4 Summary

We performed a **complete data analysis** using only the standard library:

| Module | Used for |
|--------|----------|
| `csv` | Reading/writing CSV files |
| `statistics` | mean, median, mode, stdev |
| `collections.Counter` | Frequency analysis |
| `collections.defaultdict` | Grouping & aggregation |
| `datetime` | Date parsing & monthly grouping |
| `random` | Synthetic data generation |

### Key Takeaways

1. **You don't always need pandas** — Python's stdlib is surprisingly capable
2. **Jupyter is perfect** for iterative exploration — run, see results, refine
3. **Narrative cells** help document your thinking as you analyze
4. **Synthetic data** is great for learning — you control the patterns

**Next up →** Chapter 5: Best Practices & Tips